# Afferent-Gain Model — Two-Eta Parameter Scan

Sweeps the two adaptation rates of the working afferent-gain model:

- **y-axis — recurrent Hebbian** `ETA_HEBBIAN` (on the outputs `r`; >0 facilitation)
- **x-axis — afferent gain** `ETA_AFFERENT` (on the raw input magnitude; <0 depression)

both over `[-4, -2, -1, -0.5, 0, 0.5, 1, 2, 0.5]`-style symmetric values
`[-4, -2, -1, -0.5, 0, 0.5, 1, 2, 4]` (9×9 grid). All other parameters are held
fixed, with the same dynamics-unit weight parametrisation as the example
notebook (entries `~ Normal(W_MEAN/N, W_SD/√N)`, `NORMALIZE_EV` **off**).

**Metrics per condition** (averaged over `N_NETWORKS`): the six working-regime
metrics — `lda`, `corrBin`, `actEnt`, `axisR`, `rankC`, `frac` — plus two new
display-only ones: the **mean-activity change (last − first stimulation)** for
the most **boring** sequence (column 0) and the most **max-entropy** sequence
(column 19), each averaged over the five variations.

**Plots** → `<OUTPUT_DIR>/`: one clean heatmap per metric (8 total, no regime
overlay) and the **binary black-on-white working-regime** map (black where all
six criteria hold). No criteria-count / colored-overlap / deltaEV plots.


In [ ]:
# ======================
# Afferent-Gain Model — Two-Eta Parameter Scan
# ======================

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe
from scipy.stats import spearmanr, pearsonr
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import Ridge
import os

# ================================================================
# >>>  USER SETTINGS
# ================================================================
# --- recurrent weight matrix, in DYNAMICS units (see init_W) ---
W_MEAN          =  0.25     # common-mode eigenvalue   (entry-mean * N)
W_SD            =  0.9      # random spectral radius   (entry-SD * sqrt(N))

# --- recovery time constants (sequence steps) ---
TAU_HEBBIAN     =  10.0
TAU_AFFERENT    =  10.0

# --- renormalise W_eff each step (pins mean & random radius) ---
NORMALIZE_EV    =  True    # OFF for now

# --- afferent gain clip bounds (wide so strong etas do not saturate) ---
MIN_GAIN        = -10.0
MAX_GAIN        =  10.0

# --- input structure ---
INPUT_MODE      = 'sparse_uniform'   # 'sparse_uniform' | 'dense_uniform'
SPARSITY        =  0.1

# --- display toggles ---
SHOW_REGIME     =  True     # outline the working-regime cells on every metric panel
SHOW_OUTLINE    =  True     # black frame box around every matrix
SHOW_VALUES     =  False     # print each pixel's numeric value onto the heatmap
CONTOUR_LW      =  10.0     # working-regime outline thickness
OUTLINE_LW      =  1.5      # matrix frame thickness (thin, as in the original)
VALUE_FONT_SIZE =  25       # font size of the printed pixel values

# --- the swept axes ---
eta_hebbian_values  = [-4, -2, -1, -0.5, 0, 0.5, 1, 2, 4]            # y-axis (recurrent)
eta_afferent_values = [-0.4, -0.2, -0.1, -0.05, 0, 0.05, 0.1, 0.2, 0.4]  # x-axis (feedforward; /10, far stronger effect)

N_NETWORKS      =  100      # networks per condition (set small, e.g. 3, to test)

OUTPUT_DIR      = './output_scan_afferent'
# ================================================================

# ================================================================
# Working-regime criteria  (the six; the two new metrics are display-only)
# ================================================================
# thresholds for the two activity-change criteria (signed % change, last vs first)
#   dBor = boring (min-entropy) sequence ; dEnt = max-entropy sequence
#   permissive placeholders -> tune by hand (with these defaults they never restrict)
DBOR_MIN, DBOR_MAX = -100.0, 0.0
DENT_MIN, DENT_MAX = 0.0, 100.0

CRITERIA = {
    'lda':     lambda r: r['lda']     >= 0.5,
    'corrBin': lambda r: r['corrBin'] >= 0.5,
    'actEnt':  lambda r: (r['actEnt'] >= 0.0) & (r['actEnt'] <= 1),
    'axisR':   lambda r: (r['axisR']  >= 0.0) & (r['axisR']  <= 1),
    'rankC':   lambda r: r['rankC']   >= 0.0,
    'frac':    lambda r: r['frac']    >  0.5,
    'dBor':    lambda r: (r['dBor'] >= DBOR_MIN) & (r['dBor'] <= DBOR_MAX),
    'dEnt':    lambda r: (r['dEnt'] >= DENT_MIN) & (r['dEnt'] <= DENT_MAX),
}

# ================================================================
# Style  (mirrors the original scan figures)
# ================================================================
FONT_SIZE_BASE  = 110
FONT_SIZE_TITLE = 120
FONT_SIZE_TICK  = 84
FONT_SIZE_CBAR  = 100

plt.rcParams.update({
    'font.size':        FONT_SIZE_BASE,
    'axes.titlesize':   FONT_SIZE_TITLE,
    'axes.labelsize':   FONT_SIZE_BASE,
    'xtick.labelsize':  FONT_SIZE_TICK,
    'ytick.labelsize':  FONT_SIZE_TICK,
    'figure.titlesize': FONT_SIZE_TITLE,
})

np.random.seed(42)

# ================================================================
# Simulation parameters
# ================================================================
N             = 100
stim_strength = 1.0
theta0        = 0.0
tau           = 1.0
dt            = 0.01
n_iter        = 1000

ALL_STIMULI   = [chr(ord('A') + i) for i in range(10)]
NUM_TO_LETTER = {i + 1: chr(ord('A') + i) for i in range(10)}
STIM_TO_IDX   = {s: i for i, s in enumerate(ALL_STIMULI)}

def _fmt(v):
    sign = 'm' if v < 0 else ''
    av = abs(v); ip = int(av); dp = round((av - ip) * 100)
    return f'{sign}{ip:04d}_{dp:02d}'

PARAM_TAG  = (f'afferent_wm{_fmt(W_MEAN)}_sd{_fmt(W_SD)}'
              f'_th{_fmt(TAU_HEBBIAN)}_ta{_fmt(TAU_AFFERENT)}'
              f'_ne{int(NORMALIZE_EV)}_{INPUT_MODE[:2]}{int(round(SPARSITY*100))}')
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
# MATLAB val data
# ================================================================
VAL_DATA = np.array([
    [[2,7,7,7,2,2,6,7,7,7, 8, 2, 7, 1, 2, 1, 6, 8, 9, 4],
     [2,7,2,7,7,2,2,7,6,2, 6, 5, 8, 5, 6, 6, 5, 8, 5, 7],
     [2,7,7,7,2,2,7,2,8,2, 2, 2, 5, 8, 1, 6, 6, 8, 3, 3],
     [2,2,7,7,7,2,6,2,8,2, 7, 6, 8, 7, 8, 7, 6, 2, 2,10],
     [2,2,2,7,2,2,7,7,6,8, 2, 5, 6, 5, 6, 7, 6, 8, 6, 8],
     [2,2,2,7,2,7,2,7,7,8, 6, 7, 5, 6, 6, 5, 8, 5, 1, 5],
     [2,7,2,7,6,6,7,6,8,8, 6, 2, 7, 8, 7, 5, 8, 4, 7, 6],
     [2,7,2,7,7,7,7,6,7,6, 2, 7, 6, 1, 1, 8, 3, 7, 1, 9],
     [2,2,2,2,7,7,6,6,8,2, 2, 7, 5, 1, 1, 5, 7, 6, 8, 1],
     [2,2,2,2,2,2,2,2,2,2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]],
    [[4,1,1,1,4,4, 9,1, 1, 1,10, 4, 1, 8, 4, 8, 9,10, 5, 3],
     [4,1,4,1,1,4, 4,1, 9, 4, 9, 6,10, 6, 9, 9, 6,10, 6, 1],
     [4,1,1,1,4,4, 1,4,10, 4, 4, 4, 6,10, 8, 9, 9,10, 7, 7],
     [4,4,1,1,1,4, 9,4,10, 4, 1, 9,10, 1,10, 1, 9, 4, 4, 2],
     [4,4,4,1,4,4, 1,1, 9,10, 4, 6, 9, 6, 9, 1, 9,10, 9,10],
     [4,4,4,1,4,1, 4,1, 1,10, 9, 1, 6, 9, 9, 6,10, 6, 8, 6],
     [4,1,4,1,9,9, 1,9,10,10, 9, 4, 1,10, 1, 6,10, 3, 1, 9],
     [4,1,4,1,1,1, 1,9, 1, 9, 4, 1, 9, 8, 8,10, 7, 1, 8, 5],
     [4,4,4,4,1,1, 9,9,10, 4, 4, 1, 6, 8, 8, 6, 1, 9,10, 8],
     [4,4,4,4,4,4, 4,4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]],
    [[ 6, 3, 3, 3, 6, 6,10, 3, 3, 3, 9, 6, 3, 5, 6, 5,10, 9, 7, 2],
     [ 6, 3, 6, 3, 3, 6, 6, 3,10, 6,10, 1, 9, 1,10,10, 1, 9, 1, 3],
     [ 6, 3, 3, 3, 6, 6, 3, 6, 9, 6, 6, 6, 1, 9, 5,10,10, 9, 4, 4],
     [ 6, 6, 3, 3, 3, 6,10, 6, 9, 6, 3,10, 9, 3, 9, 3,10, 6, 6, 8],
     [ 6, 6, 6, 3, 6, 6, 3, 3,10, 9, 6, 1,10, 1,10, 3,10, 9,10, 9],
     [ 6, 6, 6, 3, 6, 3, 6, 3, 3, 9,10, 3, 1,10,10, 1, 9, 1, 5, 1],
     [ 6, 3, 6, 3,10,10, 3,10, 9, 9,10, 6, 3, 9, 3, 1, 9, 2, 3,10],
     [ 6, 3, 6, 3, 3, 3, 3,10, 3,10, 6, 3,10, 5, 5, 9, 4, 3, 5, 7],
     [ 6, 6, 6, 6, 3, 3,10,10, 9, 6, 6, 3, 1, 5, 5, 1, 3,10, 9, 5],
     [ 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6]],
    [[ 8, 4, 4, 4, 8, 8, 5, 4, 4, 4, 7, 8, 4,10, 8,10, 5, 7, 2, 1],
     [ 8, 4, 8, 4, 4, 8, 8, 4, 5, 8, 5, 9, 7, 9, 5, 5, 9, 7, 9, 4],
     [ 8, 4, 4, 4, 8, 8, 4, 8, 7, 8, 8, 8, 9, 7,10, 5, 5, 7, 6, 6],
     [ 8, 8, 4, 4, 4, 8, 5, 8, 7, 8, 4, 5, 7, 4, 7, 4, 5, 8, 8, 3],
     [ 8, 8, 8, 4, 8, 8, 4, 4, 5, 7, 8, 9, 5, 9, 5, 4, 5, 7, 5, 7],
     [ 8, 8, 8, 4, 8, 4, 8, 4, 4, 7, 5, 4, 9, 5, 5, 9, 7, 9,10, 9],
     [ 8, 4, 8, 4, 5, 5, 4, 5, 7, 7, 5, 8, 4, 7, 4, 9, 7, 1, 4, 5],
     [ 8, 4, 8, 4, 4, 4, 4, 5, 4, 5, 8, 4, 5,10,10, 7, 6, 4,10, 2],
     [ 8, 8, 8, 8, 4, 4, 5, 5, 7, 8, 8, 4, 9,10,10, 9, 4, 5, 7,10],
     [ 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]],
    [[10, 9, 9, 9,10,10, 8, 9, 9, 9, 5,10, 9, 2,10, 2, 8, 5, 3, 7],
     [10, 9,10, 9, 9,10,10, 9, 8,10, 8, 4, 5, 4, 8, 8, 4, 5, 4, 9],
     [10, 9, 9, 9,10,10, 9,10, 5,10,10,10, 4, 5, 2, 8, 8, 5, 1, 1],
     [10,10, 9, 9, 9,10, 8,10, 5,10, 9, 8, 5, 9, 5, 9, 8,10,10, 6],
     [10,10,10, 9,10,10, 9, 9, 8, 5,10, 4, 8, 4, 8, 9, 8, 5, 8, 5],
     [10,10,10, 9,10, 9,10, 9, 9, 5, 8, 9, 4, 8, 8, 4, 5, 4, 2, 4],
     [10, 9,10, 9, 8, 8, 9, 8, 5, 5, 8,10, 9, 5, 9, 4, 5, 7, 9, 8],
     [10, 9,10, 9, 9, 9, 9, 8, 9, 8,10, 9, 8, 2, 2, 5, 1, 9, 2, 3],
     [10,10,10,10, 9, 9, 8, 8, 5,10,10, 9, 4, 2, 2, 4, 9, 8, 5, 2],
     [10,10,10,10,10,10,10,10,10,10,10,10,10,10,10,10,10,10,10,10]],
], dtype=int)


In [ ]:
# Data loading & entropy
# ================================================================

def load_sequences():
    sequences = []
    for s in range(5):
        for j in range(20):
            sequences.append([NUM_TO_LETTER[int(n)] for n in VAL_DATA[s, :, j]])
    return sequences


def column_sequences(col):
    """The 5 variation-sequences for one permutation column (e.g. 0=boring, 19=max-entropy)."""
    return [[NUM_TO_LETTER[int(n)] for n in VAL_DATA[s, :, col]] for s in range(5)]


def compute_entropy(sequence):
    entropies = []
    for i in range(len(sequence)):
        history = ['blank'] + sequence[:i + 1]
        u, c    = np.unique(history, return_counts=True)
        p       = c / len(history)
        entropies.append(-np.sum(p * np.log2(p + 1e-12)))
    return entropies


In [ ]:
# ================================================================
# Init / stimuli / normalisation  (same as the example notebook)
# ================================================================

def relu(x):
    return np.maximum(0, x)


def renormalize_W_eff(M):
    """Reset random part to spectral radius W_SD (entry-SD = W_SD/sqrt(N)) while
    pinning the mean to the defined common-mode eigenvalue W_MEAN (entry-mean =
    W_MEAN/N). No epsilon, no eigenvalues; zero-variance is an early return."""
    mu          = M.mean()
    sd          = M.std()
    target_mean = W_MEAN / N
    if sd == 0:
        return np.full_like(M, target_mean)
    return (M - mu) / sd * (W_SD / np.sqrt(N)) + target_mean


def init_W():
    """Gaussian matrix normalised EXACTLY: entry-mean = W_MEAN/N, entry-SD =
    W_SD/sqrt(N) -> common-mode eigenvalue W_MEAN, random radius W_SD."""
    return renormalize_W_eff(np.random.normal(0.0, 1.0, (N, N)))


def create_stimuli():
    stimuli  = {}
    n_active = max(1, int(round(SPARSITY * N)))
    for name in ALL_STIMULI:
        if INPUT_MODE == 'sparse_uniform':
            I      = np.zeros(N)
            idx    = np.random.choice(N, n_active, replace=False)
            I[idx] = np.random.uniform(0.0, 1.0, n_active)
        elif INPUT_MODE == 'dense_uniform':
            I = np.random.uniform(0.0, 1.0, N)
        else:
            raise ValueError(f'unknown INPUT_MODE: {INPUT_MODE}')
        stimuli[name] = I * stim_strength
    return stimuli


In [ ]:
# ================================================================
# Core sequence runner  —  eta_heb (recurrent Hebbian on r) + eta_aff (afferent gain on I_raw)
# ================================================================

def run_sequence(seq, W, stimuli, eta_heb, eta_aff):
    eff       = np.ones((N, N))
    g         = np.ones(N)
    responses = []
    rec_h = np.exp(-1.0 / TAU_HEBBIAN)
    rec_g = np.exp(-1.0 / TAU_AFFERENT)

    for stim_name in seq:
        I_raw = stimuli[stim_name]
        I     = g * I_raw
        r     = np.zeros(N)

        W_eff = W * eff
        if NORMALIZE_EV:
            W_eff = renormalize_W_eff(W_eff)

        for _ in range(n_iter):
            r += dt * (-r + relu(W_eff @ r + I - theta0)) / tau

        responses.append(r.copy())

        r_sig = np.nan_to_num(r, nan=0.0, posinf=0.0, neginf=0.0)
        r_sig = np.clip(r_sig, -1e6, 1e6)

        if eta_heb != 0:
            denom = N * np.mean(r_sig ** 2)
            if denom > 0:
                eff += eta_heb * np.outer(r_sig, r_sig) / denom
            eff = 1.0 + (eff - 1.0) * rec_h
            eff = np.clip(eff, -1e6, 1e6)

        if eta_aff != 0:
            g = g + eta_aff * I_raw
            g = 1.0 + (g - 1.0) * rec_g
            g = np.clip(g, MIN_GAIN, MAX_GAIN)

    return np.array(responses)


In [ ]:
# ================================================================
# Binning & metrics helpers
# ================================================================
def bin_responses_by_entropy(all_responses, all_entropy, all_identity, n_bins=5):
    emin, emax = all_entropy.min(), all_entropy.max()
    edges      = np.linspace(emin, emax, n_bins + 1)
    bidx       = np.clip(np.digitize(all_entropy, edges[:-1]) - 1, 0, n_bins - 1)
    bin_means, bin_ent_means = {}, {}
    for b in range(n_bins):
        mask = bidx == b
        if mask.sum() == 0:
            continue
        bin_ent_means[b] = float(np.mean(all_entropy[mask]))
        bin_means[b] = {}
        for stim in np.unique(all_identity[mask]):
            sm = mask & (all_identity == stim)
            if sm.sum() > 0:
                bin_means[b][stim] = np.mean(all_responses[sm], axis=0)
    return bin_means, bin_ent_means


def _rsm_upper_tri(responses_dict):
    stim_responses = [responses_dict[s] for s in ALL_STIMULI if s in responses_dict]
    if len(stim_responses) < 2:
        return None
    mat = np.array(stim_responses)
    if np.std(mat) < 1e-10:
        return None
    C = np.nan_to_num(np.corrcoef(mat), nan=0.0)
    return C[np.triu_indices(C.shape[0], k=1)]


def compute_LDA_discriminability(R, H, n_splits=5):
    R = np.nan_to_num(R, nan=0.0)
    labels = (H > np.median(H)).astype(int)
    if len(np.unique(labels)) < 2:
        return 0.5
    n_comp = min(20, R.shape[0] - 1, R.shape[1])
    if n_comp < 1:
        return 0.5
    X   = PCA(n_components=n_comp).fit_transform(R)
    lda = LinearDiscriminantAnalysis()
    cv  = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    accs = []
    for tr, te in cv.split(X, labels):
        try:
            lda.fit(X[tr], labels[tr])
            accs.append(np.mean(lda.predict(X[te]) == labels[te]))
        except Exception:
            accs.append(0.5)
    return float(np.mean(accs))


def compute_R_meanCorrOverBins(R, H, ID):
    R = np.nan_to_num(R, nan=0.0)
    bm, _ = bin_responses_by_entropy(R, H, ID)
    uts = [_rsm_upper_tri(bm[b]) for b in sorted(bm) if _rsm_upper_tri(bm[b]) is not None]
    if len(uts) < 2:
        return 0.0
    pairs = []
    for i in range(len(uts)):
        for j in range(i + 1, len(uts)):
            a, b = uts[i], uts[j]
            if np.std(a) > 1e-10 and np.std(b) > 1e-10:
                try:
                    r, _ = pearsonr(a, b)
                    if not np.isnan(r):
                        pairs.append(r)
                except Exception:
                    pass
    return float(np.mean(pairs)) if pairs else 0.0


def compute_R_meanAct_entropy(R, H, n_bins=5):
    R = np.nan_to_num(R, nan=0.0)
    edges = np.linspace(H.min(), H.max(), n_bins + 1)
    bidx  = np.clip(np.digitize(H, edges[:-1]) - 1, 0, n_bins - 1)
    acts, ents = [], []
    for b in range(n_bins):
        mask = bidx == b
        if mask.sum() == 0:
            continue
        acts.append(float(np.mean(R[mask])))
        ents.append(float(np.mean(H[mask])))
    if len(acts) < 2:
        return 0.0
    try:
        r, _ = pearsonr(acts, ents)
        return float(r) if not np.isnan(r) else 0.0
    except Exception:
        return 0.0


def compute_R_entropyAxisVsMeanActAxis(R, H):
    R = np.nan_to_num(R, nan=0.0)
    if np.std(R) < 1e-10:
        return 0.0
    ma = np.mean(R, axis=1)
    try:
        ridge = Ridge(alpha=1.0)
        ridge.fit(R, H);  we = ridge.coef_
        ridge.fit(R, ma); wa = ridge.coef_
        if np.std(we) > 1e-10 and np.std(wa) > 1e-10:
            r, _ = pearsonr(we, wa)
            return float(r) if not np.isnan(r) else 0.0
    except Exception:
        pass
    return 0.0


def compute_R_meanCorrVsEntropy(R, H, ID, n_bins=5):
    R = np.nan_to_num(R, nan=0.0)
    bm, bem = bin_responses_by_entropy(R, H, ID, n_bins)
    mc, be = [], []
    for b in sorted(bm.keys()):
        ut = _rsm_upper_tri(bm[b])
        if ut is not None and b in bem:
            mc.append(float(np.mean(ut)))
            be.append(bem[b])
    if len(mc) < 2:
        return 0.0
    try:
        rho, _ = spearmanr(mc, be)
        return float(rho) if not np.isnan(rho) else 0.0
    except Exception:
        return 0.0


def compute_fractPositiveEntropyTuning(R, H):
    R = np.nan_to_num(R, nan=0.0)
    pos = 0
    for i in range(N):
        if np.std(R[:, i]) > 0 and np.std(H) > 0:
            try:
                rho, _ = spearmanr(R[:, i], H)
                if not np.isnan(rho) and rho > 0:
                    pos += 1
            except Exception:
                pass
    return pos / N


In [ ]:
# ================================================================
# Single experiment  (6 metrics + 2 new last-first activity metrics)
# ================================================================

def _pct_change_last_first(col, W, stimuli, eta_heb, eta_aff):
    """Signed percentage change in mean activity, last vs first stimulation,
    averaged over the 5 variations. Variations whose first-step activity is
    ~0 (no baseline to divide by) are skipped."""
    pcts = []
    for seq in column_sequences(col):
        resps = run_sequence(seq, W, stimuli, eta_heb, eta_aff)
        rs    = np.nan_to_num(resps, nan=0.0, posinf=0.0, neginf=0.0)
        first = float(np.mean(rs[0]))
        last  = float(np.mean(rs[-1]))
        if abs(first) > 1e-9:
            pcts.append((last - first) / first * 100.0)
    return float(np.mean(pcts)) if pcts else 0.0


def run_single_experiment(eta_heb, eta_aff):
    W       = init_W()
    stimuli = create_stimuli()

    all_R, all_H, all_ID = [], [], []
    for seq in load_sequences():
        ents  = compute_entropy(seq)
        resps = run_sequence(seq, W, stimuli, eta_heb, eta_aff)
        all_R.extend(resps); all_H.extend(ents); all_ID.extend(seq)

    R  = np.array(all_R); H = np.array(all_H); ID = np.array(all_ID)

    return dict(
        lda     = compute_LDA_discriminability(R, H),
        corrBin = compute_R_meanCorrOverBins(R, H, ID),
        actEnt  = compute_R_meanAct_entropy(R, H),
        axisR   = compute_R_entropyAxisVsMeanActAxis(R, H),
        rankC   = compute_R_meanCorrVsEntropy(R, H, ID),
        frac    = compute_fractPositiveEntropyTuning(R, H),
        dBor    = _pct_change_last_first(0,  W, stimuli, eta_heb, eta_aff),   # most boring
        dEnt    = _pct_change_last_first(19, W, stimuli, eta_heb, eta_aff),   # most max-entropy
    )


# ================================================================
# Parameter scan over (eta_hebbian x eta_afferent)
# ================================================================

def run_scan():
    n_h  = len(eta_hebbian_values)
    n_a  = len(eta_afferent_values)
    keys = ['lda', 'corrBin', 'actEnt', 'axisR', 'rankC', 'frac', 'dBor', 'dEnt']
    res  = {k: np.zeros((n_h, n_a)) for k in keys}

    total = n_h * n_a * N_NETWORKS
    current = 0
    print('=' * 65)
    print('Afferent-Gain Two-Eta Scan')
    print(f'  grid: {n_h} (Hebbian) x {n_a} (afferent)  |  N_NETWORKS={N_NETWORKS}'
          + ('  [TEST MODE]' if N_NETWORKS <= 5 else ''))
    print(f'  NORMALIZE_EV={NORMALIZE_EV}  INPUT_MODE={INPUT_MODE}')
    print(f'  total runs: {total}')
    print('=' * 65)

    for i, eh in enumerate(eta_hebbian_values):
        for j, ea in enumerate(eta_afferent_values):
            print(f'\nCondition ({i+1},{j+1}): eta_heb={eh}, eta_aff={ea}')
            accum = {k: [] for k in keys}
            for net in range(N_NETWORKS):
                current += 1
                print(f'  Network {net+1}/{N_NETWORKS}  (run {current}/{total})')
                vals = run_single_experiment(eh, ea)
                for k in keys:
                    accum[k].append(vals[k])
            for k in keys:
                res[k][i, j] = float(np.mean(accum[k]))
    return res


def compute_working_regime(res):
    mask = np.ones(res['lda'].shape, dtype=bool)
    for fn in CRITERIA.values():
        mask &= fn(res)
    return mask


In [ ]:
# ================================================================
# Plot layout & panels
# ================================================================

N_ROWS = len(eta_hebbian_values)    # y
N_COLS = len(eta_afferent_values)   # x

XLABEL = '\u0394 afferent adapt'
YLABEL = '\u0394 recurrent Hebb'

FIG_W = 60
FIG_H = 44
MAT_H_FRAC = 1.0 / 3.0
MAT_H_IN   = FIG_H * MAT_H_FRAC
MAT_W_IN   = MAT_H_IN
MAT_W_FRAC = MAT_W_IN / FIG_W
AX_LEFT   = 0.18
AX_BOTTOM = 0.26
AX_RECT   = [AX_LEFT, AX_BOTTOM, MAT_W_FRAC, MAT_H_FRAC]
CBAR_GAP    = 0.015
CBAR_W_FRAC = 0.018
CBAR_RECT   = [AX_LEFT + MAT_W_FRAC + CBAR_GAP, AX_BOTTOM, CBAR_W_FRAC, MAT_H_FRAC]

# --- display: the two activity-change panels (signed % change) ---
PCT_CAP       = 100.0   # colour-axis cap (+/- %) for dBor / dEnt
PCT_LINTHRESH = 1.0     # symlog linear threshold (% within which the scale is linear)

PANELS = [
    # (key,    title,                              cmap,     vmin, vmax, cbar_label)
    ('lda',     'Entropy Decoding',                'RdYlGn',  0.0,  1.0, 'Fraction correct'),
    ('corrBin', 'Representational Stability',      'RdBu_r', -1,    1,   'Pearson r'),
    ('actEnt',  'Corr (Activity, Entropy)',        'RdBu_r', -1,    1,   'Pearson r'),
    ('axisR',   'Corr (Act. Axis, Entr. Axis)',    'RdBu_r', -1,    1,   'Pearson r'),
    ('rankC',   'Rank Corr\n(Rep. Sim, Entr.)',    'RdBu_r', -1,    1,   'Spearman \u03c1'),
    ('frac',    'Entropy Tuning',                  'PRGn',    0,    1,   'Fraction'),
    ('dBor',    'Mean Act: Boring\n(% change last vs first)',     'RdBu_r', -PCT_CAP, PCT_CAP, '% change'),
    ('dEnt',    'Mean Act: Max-Entropy\n(% change last vs first)','RdBu_r', -PCT_CAP, PCT_CAP, '% change'),
]


def _make_fig_and_axes():
    fig = plt.figure(figsize=(FIG_W, FIG_H))
    ax  = fig.add_axes(AX_RECT)
    cax = fig.add_axes(CBAR_RECT)
    return fig, ax, cax


def _decorate_axes(ax, title):
    ax.set_xticks(range(N_COLS))
    ax.set_yticks(range(N_ROWS))
    ax.set_xticklabels([str(v) for v in eta_afferent_values],
                       fontsize=FONT_SIZE_TICK, rotation=45, ha='right',
                       rotation_mode='anchor')
    ax.set_yticklabels([str(v) for v in eta_hebbian_values], fontsize=FONT_SIZE_TICK)
    ax.set_xlabel(XLABEL, fontsize=FONT_SIZE_BASE, labelpad=20)
    ax.set_ylabel(YLABEL, fontsize=FONT_SIZE_BASE, labelpad=20)
    ax.set_title(title, fontsize=FONT_SIZE_TITLE, fontweight='bold', pad=28)
    ax.grid(False)


def _draw_outline(ax):
    """Thin black frame around the matrix, as in the original notebooks
    (toggle SHOW_OUTLINE)."""
    for spine in ax.spines.values():
        spine.set_visible(SHOW_OUTLINE)
        if SHOW_OUTLINE:
            spine.set_linewidth(OUTLINE_LW)


def _annotate_values(ax, data, fmt='{:.3g}'):
    """Print each pixel's numeric value on the heatmap (toggle SHOW_VALUES).
    Black text with a white halo so it stays readable on any cell colour."""
    if not SHOW_VALUES:
        return
    nrows, ncols = data.shape
    for i in range(nrows):
        for j in range(ncols):
            v = data[i, j]
            if not np.isfinite(v):
                continue
            t = ax.text(j, i, fmt.format(v), ha='center', va='center',
                        fontsize=VALUE_FONT_SIZE, color='black', zorder=6)
            t.set_path_effects([pe.withStroke(linewidth=3, foreground='white')])


def _add_working_regime_contour(ax, mask):
    """Pixel-exact black outline around every working-regime cell, drawn the
    way the original notebooks did (clip_on=False so border segments show).
    Toggle SHOW_REGIME."""
    if not SHOW_REGIME or not np.any(mask):
        return
    nrows, ncols = mask.shape
    kw = dict(color='black', lw=CONTOUR_LW, solid_capstyle='butt',
              clip_on=False, zorder=8)
    for i in range(nrows):
        for j in range(ncols):
            if not mask[i, j]:
                continue
            if i == 0 or not mask[i - 1, j]:
                ax.plot([j - 0.5, j + 0.5], [i - 0.5, i - 0.5], **kw)
            if i == nrows - 1 or not mask[i + 1, j]:
                ax.plot([j - 0.5, j + 0.5], [i + 0.5, i + 0.5], **kw)
            if j == 0 or not mask[i, j - 1]:
                ax.plot([j - 0.5, j - 0.5], [i - 0.5, i + 0.5], **kw)
            if j == ncols - 1 or not mask[i, j + 1]:
                ax.plot([j + 0.5, j + 0.5], [i - 0.5, i + 0.5], **kw)

In [ ]:
# ================================================================
# Plots:  clean per-metric heatmaps  +  binary black-on-white region
# ================================================================

def save_individual_plots(res, working_regime):
    for key, title, cmap, vmin, vmax, cbar_label in PANELS:
        data = res[key]
        fig, ax, cax = _make_fig_and_axes()

        if key in ('dBor', 'dEnt'):
            # signed % change: capped at +-PCT_CAP, symmetric-log colour scale
            norm = mcolors.SymLogNorm(linthresh=PCT_LINTHRESH, vmin=vmin, vmax=vmax, base=10)
            im = ax.imshow(data, cmap=cmap, norm=norm, aspect='equal', origin='lower')
        else:
            if vmin is None or vmax is None:           # diverging, symmetric about 0
                am = np.max(np.abs(data[np.isfinite(data)])) if np.any(np.isfinite(data)) else 1.0
                am = am if am > 0 else 1.0
                vmin_, vmax_ = -am, am
            else:
                vmin_, vmax_ = vmin, vmax
            im = ax.imshow(data, cmap=cmap, vmin=vmin_, vmax=vmax_,
                           aspect='equal', origin='lower')

        _decorate_axes(ax, title)
        _add_working_regime_contour(ax, working_regime)
        _annotate_values(ax, data)
        _draw_outline(ax)
        cbar = fig.colorbar(im, cax=cax)
        cbar.set_label(cbar_label, fontsize=FONT_SIZE_CBAR, labelpad=16)
        cbar.ax.tick_params(labelsize=FONT_SIZE_TICK)

        base = os.path.join(OUTPUT_DIR, f'scan_{PARAM_TAG}_{key}')
        fig.savefig(base + '.png', dpi=150, bbox_inches='tight', pad_inches=1.5)
        fig.savefig(base + '.eps', format='eps', bbox_inches='tight', pad_inches=1.5)
        plt.close()
        print(f'  saved  scan_{PARAM_TAG}_{key}.png  +  .eps')


def save_summary_binary_plot(working_regime):
    fig = plt.figure(figsize=(FIG_W, FIG_H))
    ax  = fig.add_axes(AX_RECT)
    ax.imshow(working_regime.astype(float), cmap='gray_r', vmin=0, vmax=1,
              aspect='equal', origin='lower')
    _decorate_axes(ax, '')
    _annotate_values(ax, working_regime.astype(float), fmt='{:.0f}')
    _draw_outline(ax)
    base = os.path.join(OUTPUT_DIR, f'scan_{PARAM_TAG}_working_regime')
    fig.savefig(base + '.png', dpi=150, bbox_inches='tight', pad_inches=6)
    fig.savefig(base + '.eps', format='eps', bbox_inches='tight', pad_inches=6)
    plt.close()
    print(f'  saved  scan_{PARAM_TAG}_working_regime.png  +  .eps')


In [ ]:
# ================================================================
# Main
# ================================================================
if __name__ == '__main__':
    results = run_scan()

    print('\n' + '=' * 65)
    working_regime = compute_working_regime(results)
    n_good = int(np.sum(working_regime))
    print(f'  {n_good} / {working_regime.size} pixels satisfy all 6 criteria')
    print('Saving plots …')
    print('=' * 65)

    save_individual_plots(results, working_regime)
    save_summary_binary_plot(working_regime)

    np.savez(os.path.join(OUTPUT_DIR, f'results_{PARAM_TAG}.npz'),
             **results,
             eta_hebbian_values=eta_hebbian_values,
             eta_afferent_values=eta_afferent_values,
             working_regime=working_regime)

    print(f'\nAll done. Figures saved to: {os.path.abspath(OUTPUT_DIR)}')
